# SentinelIQ — 01 Data Exploration
Explore all three simulated data streams: logs, metrics, and network flows.

In [ ]:
# Setup
!git clone https://github.com/hasan-rajab/SentinelIQ.git 2>/dev/null || echo "Already cloned"
%cd /kaggle/working/SentinelIQ
import sys
sys.path.insert(0, '/kaggle/working/SentinelIQ')


In [ ]:
# Generate datasets
!python data/simulated/pipeline.py --duration 120 --anomaly-rate 0.08


In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="darkgrid")

def load_jsonl(path):
    return pd.DataFrame([json.loads(l) for l in open(path)])

logs_df    = load_jsonl('data/simulated/logs.jsonl')
metrics_df = load_jsonl('data/simulated/metrics.jsonl')
network_df = load_jsonl('data/simulated/network.jsonl')

print(f"Logs    : {len(logs_df):,} records | {logs_df['is_anomaly'].sum()} anomalies")
print(f"Metrics : {len(metrics_df):,} records | {metrics_df['is_anomaly'].sum()} anomalies")
print(f"Network : {len(network_df):,} records | {network_df['is_anomaly'].sum()} anomalies")


## Logs Dataset

In [ ]:
logs_df.head(5)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Anomaly distribution
logs_df['is_anomaly'].value_counts().plot(kind='bar', ax=axes[0], color=['#2ecc71','#e74c3c'])
axes[0].set_title('Log Anomaly Distribution')
axes[0].set_xticklabels(['Normal', 'Anomaly'], rotation=0)

# Anomaly types
anomaly_types = logs_df[logs_df['is_anomaly']]['anomaly_type'].value_counts()
anomaly_types.plot(kind='barh', ax=axes[1], color='#e74c3c')
axes[1].set_title('Log Anomaly Types')

plt.tight_layout()
plt.show()


In [ ]:
# Log type breakdown
print("Log type distribution:")
print(logs_df['log_type'].value_counts())
print()
print("Sample anomalous log messages:")
for _, row in logs_df[logs_df['is_anomaly']].head(5).iterrows():
    print(f"  [{row['anomaly_type']:25}] {row['message'][:80]}")


## Metrics Dataset

In [ ]:
metrics_df.head(5)


In [ ]:
feature_cols = ['cpu_percent','mem_percent','disk_read_mbps',
                'disk_write_mbps','net_in_mbps','net_out_mbps',
                'open_connections','process_count']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    normal = metrics_df[~metrics_df['is_anomaly']][col]
    anomaly = metrics_df[metrics_df['is_anomaly']][col]
    axes[i].hist(normal, bins=30, alpha=0.6, color='#2ecc71', label='Normal')
    axes[i].hist(anomaly, bins=30, alpha=0.6, color='#e74c3c', label='Anomaly')
    axes[i].set_title(col)
    axes[i].legend(fontsize=8)

plt.suptitle('Metric Feature Distributions: Normal vs Anomaly', fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
print("Metrics anomaly types:")
print(metrics_df[metrics_df['is_anomaly']]['anomaly_type'].value_counts())
print()
print("Feature statistics (normal vs anomaly):")
print(metrics_df.groupby('is_anomaly')[feature_cols].mean().T.rename(columns={False:'Normal Mean', True:'Anomaly Mean'}))


## Network Dataset

In [ ]:
network_df.head(5)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Anomaly distribution
network_df['is_anomaly'].value_counts().plot(kind='bar', ax=axes[0], color=['#2ecc71','#e74c3c'])
axes[0].set_title('Network Anomaly Distribution')
axes[0].set_xticklabels(['Normal','Anomaly'], rotation=0)

# Anomaly types
network_df[network_df['is_anomaly']]['anomaly_type'].value_counts().plot(
    kind='barh', ax=axes[1], color='#e74c3c')
axes[1].set_title('Network Anomaly Types')

# Protocol distribution
network_df['protocol'].value_counts().plot(kind='pie', ax=axes[2], autopct='%1.1f%%')
axes[2].set_title('Protocol Distribution')

plt.tight_layout()
plt.show()


In [ ]:
net_features = ['bytes_out','bytes_in','packets','duration_ms']
print("Network feature stats (normal vs anomaly):")
print(network_df.groupby('is_anomaly')[net_features].mean().T.rename(
    columns={False:'Normal Mean', True:'Anomaly Mean'}))

print()
print("Top destination ports:")
print(network_df['dst_port'].value_counts().head(10))


## Correlation Analysis (Metrics)

In [ ]:
corr = metrics_df[feature_cols].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Metric Feature Correlation Matrix')
plt.tight_layout()
plt.show()


## Summary
- All three data streams are generating correctly
- Anomaly rates are ~8% as configured
- Feature distributions show clear separation between normal and anomaly classes
- Ready for Phase 2 model training